# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure mlcroissant library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

In [ ]:
# List available RecordSets, Fields, and Columns (by their @id)
record_sets = []
if hasattr(metadata, 'record_sets'):
    record_sets = metadata.record_sets
else:
    # Attempt to find record_sets attribute (for backward compatibility)
    if hasattr(metadata, 'recordSet'):
        record_sets = metadata.recordSet

print("Available Record Sets (@id):")
for record_set in record_sets:
    print(f"  - {getattr(record_set, '@id', None)} | {getattr(record_set, 'name', '')}")


# For each record set, list the available fields and columns (@id)
for record_set in record_sets:
    print(f"\nFields for RecordSet '@id': {getattr(record_set, '@id', None)}")
    fields = getattr(record_set, 'fields', [])
    for field in fields:
        print(f"  Field: {getattr(field, '@id', None)} | {getattr(field, 'name', '')} | Data type: {getattr(field, 'data_type', '')}")
        columns = getattr(field, 'columns', [])
        if columns:
            for column in columns:
                print(f"     Column: {getattr(column, '@id', None)} | {getattr(column, 'name', '')}")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# Select the record set(s) to load (by @id):
record_set_ids = []
for record_set in record_sets:
    if hasattr(record_set, '@id'):
        record_set_ids.append(getattr(record_set, '@id'))
print("Record set IDs found:", record_set_ids)

# For illustration, select the first record set, or specify one manually if multiple are present.
selected_record_set_id = record_set_ids[0] if record_set_ids else None

dataframes = {}
for record_set_id in record_set_ids:
    try:
        records = list(dataset.records(record_set=record_set_id))
        if len(records) > 0:
            df = pd.DataFrame(records)
            dataframes[record_set_id] = df
            print(f"Loaded DataFrame for RecordSet '@id': {record_set_id}. Shape: {df.shape}")
        else:
            print(f"No records found in record set '@id': {record_set_id}")
    except Exception as e:
        print(f"Error loading records for record set '@id': {record_set_id} | {e}")

# Display columns from the first loaded DataFrame
if selected_record_set_id in dataframes:
    print(f"Columns: {dataframes[selected_record_set_id].columns.tolist()}")
    display(dataframes[selected_record_set_id].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section should include operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

In [ ]:
import numpy as np

# Pick a DataFrame for EDA
df = dataframes[selected_record_set_id] if selected_record_set_id and selected_record_set_id in dataframes else None

# Show available numeric columns for selection
print("Numeric columns in this record set:")
numeric_columns = []
if df is not None:
    numeric_columns = df.select_dtypes(include=np.number).columns.tolist()
    print(numeric_columns)

# User selects one numeric column (by @id or name)
if len(numeric_columns) > 0:
    numeric_field = numeric_columns[0]
    print(f"Selecting '{numeric_field}' as the field for numeric analysis.")
    # Set a threshold value (for demonstration)
    threshold = df[numeric_field].mean() if np.issubdtype(df[numeric_field].dtype, np.number) else 10
    filtered_df = df[df[numeric_field] > threshold]
    print(f"Filtered records with {numeric_field} > {threshold}:")
    display(filtered_df.head())

    # Normalize
    filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    print(f"Normalized {numeric_field} for filtered records:")
    display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

    # Group by another column if present
    group_candidates = [col for col in df.columns if col not in numeric_columns and df[col].dtype == object]
    if len(group_candidates) > 0:
        group_field = group_candidates[0]
        print(f"Grouping by field: {group_field}")
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
        display(grouped_df.head())
else:
    print("No numeric fields found to perform EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
%matplotlib inline

if df is not None and len(numeric_columns) > 0:
    plt.figure(figsize=(8, 4))
    df[numeric_field].hist(bins=30)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel("Frequency")
    plt.show()
    
    # Boxplot for outlier analysis
    plt.figure(figsize=(6,3))
    df.boxplot(column=numeric_field)
    plt.title(f"Boxplot of {numeric_field}")
    plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

*In this notebook, we explored the FAIR^2 dataset: Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells. Using `mlcroissant`, we loaded the dataset's metadata, reviewed the record sets and their fields (referenced by their `@id`), extracted tabular data for analysis, applied some basic exploratory data analysis and normalization, and visualized distributions of key numeric fields. Further analysis can be tailored to specific research questions regarding protein expression and EV biomarker discovery.*